In [1]:
from abc import abstractmethod
from sys import path
from typing import List, Callable

import numpy as np
from torch import Tensor
import torch
from torch import nn
from torch.nn import Module as TorchModule
from transformers import AutoTokenizer, AutoModelForCausalLM
from dotenv import load_dotenv
import os
import matplotlib.pyplot as plt
from matplotlib.ticker import SymmetricalLogLocator, MaxNLocator

load_dotenv()

class Module():
    def __init__(self, module_name: str, module_data: TorchModule, module_index: int = None) -> None:
        self.module_name = module_name
        self.module_data = module_data
        self.module_index = module_index

    def get_weights(self) -> Tensor:
        if hasattr(self.module_data, "weight"):
            return self.module_data.weight.detach().data.float().cpu().numpy().flatten()
        else:
            raise Exception(f"Module '{self.module_name}' does not have weights.")
        
    def has_weights(self) -> bool:
        return hasattr(self.module_data, "weight")
    

class Model():
    def __init__(self, model_name: str) -> None:
        self.model_name = model_name
        self.tokenizer = None
        self.model = None
        self.modules: List[Module] = []
        
    def load_from_hf(self, token: str) -> None:
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name, token=token)
        self.model = AutoModelForCausalLM.from_pretrained(self.model_name, token=token)
        self.model.eval()
        
        self.modules: List[Module] = []
        for i, (name, module) in enumerate(self.model.named_modules()):
            layer = Module(name, module, module_index=i)
            self.modules.append(layer)
            
    def load_from_disk(self, path: str) -> None:
        self.tokenizer = AutoTokenizer.from_pretrained(path)
        self.model = AutoModelForCausalLM.from_pretrained(path)
        self.model.eval()

        self.modules = []
        for i, (name, module) in enumerate(self.model.named_modules()):
            self.modules.append(Module(name, module, module_index=i))
        
    def save(self, path: str) -> None:
        if self.model is None or self.tokenizer is None:
            raise Exception("Model and tokenizer must be loaded before saving.")
        
        os.makedirs(path, exist_ok=True)
        self.tokenizer.save_pretrained(path)
        self.model.save_pretrained(path)
            
    def replace_module(self, name: str, new_mod: nn.Module) -> None:
        parts = name.split(".")
        parent = self.model
        for part in parts[:-1]:
            parent = getattr(parent, part)
        setattr(parent, parts[-1], new_mod)
        
    def generate(self, prompt: str, max_new_tokens: int = 200) -> str:
        if self.model is None or self.tokenizer is None:
            raise Exception("Model and tokenizer must be loaded before generation.")
        
        inputs = self.tokenizer(prompt, return_tensors="pt")
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
            )

        new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
        return self.tokenizer.decode(new_tokens, skip_special_tokens=True)

/home/user/LLMs/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class Analyzer():
    """
        The analyzer acts as a container for the collected weights and provides methods to collect and retrieve them.
    """
    
    def __init__(self) -> None:
        self.collected_weights: dict[Model, dict[str, np.ndarray]] = {}
        
    def collect_weights(self, model: Model, reduction: float, min_count: int | None = None, max_count: int | None = None) -> None:
        if not 0 < reduction <= 1:
            raise ValueError("Reduction must be in the interval (0, 1].")
        
        module_weights: dict[str, np.ndarray] = {}
        for module in model.modules:
            if module.has_weights():
                layer_weights = module.get_weights()
                target_size = max(1, int(len(layer_weights) * reduction))
                min_count = min_count if min_count is not None else 0
                max_count = max_count if max_count is not None else len(layer_weights)
                # min and max should act as clamping values
                # make sure target_size is at least min_count and at most max_count
                min_count = min(min_count, len(layer_weights))
                max_count = min(max_count, len(layer_weights))
                
                target_size = max(min_count, min(target_size, max_count))
                
                indices = np.random.choice(len(layer_weights), size=target_size, replace=False)
                module_weights[module.module_name] = layer_weights[indices]
            
        self.collected_weights[model] = module_weights

    def get_module_weights(self, model: Model, module_name: str) -> np.ndarray:
        if model not in self.collected_weights:
            raise Exception(f"Weights not collected for model {model.model_name}. Operation aborted.")
        if module_name not in self.collected_weights[model]:
            raise Exception(f"Module '{module_name}' not found or has no weights.")
        return self.collected_weights[model][module_name]
    
    def get_all_module_weights(self, model: Model) -> dict[str, np.ndarray]:
        if model not in self.collected_weights:
            raise Exception(f"Weights not collected for model {model.model_name}. Operation aborted.")
        return self.collected_weights[model]

    def get_all_weights(self, model: Model) -> np.ndarray:
        if model not in self.collected_weights:
            raise Exception(f"Weights not collected for model {model.model_name}. Operation aborted.")
        return np.concatenate(list(self.collected_weights[model].values()))

In [3]:
class Display():
    def __init__(self) -> None:
        pass
        
    def show_all_distribution(self, weights: np.ndarray, log_y: bool = False) -> None:
        plt.figure(figsize=(12, 6))
        plt.hist(weights, bins=100)
        plt.title("Distribution of weights from all layers")
        plt.xlabel("Weight Value")
        plt.ylabel("Frequency")
        if log_y:
            plt.yscale("log")
        # plt.xscale("symlog", linthresh=3.0)
        ax = plt.gca()
        ax.xaxis.set_major_locator(MaxNLocator(nbins='auto', integer=False))
        plt.tight_layout()
        plt.show()
        
    def show_layer_distributions(self, layer_weights: dict[str, np.ndarray], log_y: bool = False) -> None:
        n = len(layer_weights)
        fig, axes = plt.subplots(n, 1, figsize=(12, n * 1.5), sharex=True)
        
        if n == 1:
            axes = [axes]
        
        all_weights = np.concatenate(list(layer_weights.values()))
        x_min, x_max = all_weights.min(), all_weights.max()
        
        for ax, (name, weights) in zip(axes, layer_weights.items()):
            ax.hist(weights, bins=80, color="steelblue", alpha=0.8)
            ax.set_xlim(x_min, x_max)
            ax.set_ylabel(name, fontsize=6, rotation=0, labelpad=4, ha='right', va='center')
            ax.yaxis.set_major_locator(MaxNLocator(nbins=3, integer=True))
            ax.tick_params(axis='both', labelsize=6)
            if log_y:
                ax.set_yscale("log")
        
        axes[-1].set_xlabel("Weight Value")
        fig.suptitle("Weight distributions by layer (top → bottom)", fontsize=12)
        plt.tight_layout()
        plt.show()

In [4]:
class Rule():
    def __init__(self, name: str, condition: Callable[[Module], bool], 
                 action: Callable[[Module, Model], None], allow_rule_stacking: bool = False) -> None:
        self.name = name
        self.condition = condition
        self.action = action
        self.allow_rule_stacking = allow_rule_stacking

    def evaluate(self, module: Module) -> bool:
        return self.condition(module)

    def apply(self, module: Module, model: Model) -> None:
        self.action(module, model)
        

class Quantizer():
    def __init__(self) -> None:
        self.rules: List[Rule] = []
        
    def add_rule(self, rule: Rule) -> None:
        self.rules.append(rule)
        
    def add_rules(self, rules: List[Rule]) -> None:
        self.rules.extend(rules)
        
    def apply(self, model: Model) -> None:
        for module in model.modules:
            for rule in self.rules:
                if rule.evaluate(module):
                    rule.apply(module, model)

                    if not rule.allow_rule_stacking:
                        break

# Abstractions end here, below are examples

In [5]:
class Int8Linear(nn.Module):
    def __init__(self, w_int8: torch.Tensor, scale: torch.Tensor, bias=None):
        super().__init__()
        self.register_buffer("weight", w_int8) 
        self.register_buffer("scale", scale)
        self.bias = bias

    def forward(self, x):
        w = self.weight.to(x.dtype) * self.scale 
        return nn.functional.linear(x, w, self.bias)

    
def action_quantize_int8(module: Module, model: Model) -> None:
    """ 
        Replace nn.Linear with Int8Linear
    
        Uses basic quantization of remapping everything to int8 with no clustering optimizations.
    """
    mod = module.module_data
    if not isinstance(mod, nn.Linear):
        raise Exception(f"Module '{module.module_name}' is not an nn.Linear. Operation aborted.")

    w = mod.weight.data
    scale = (w.abs().max() / 127.0).to(torch.float16)
    w_int8 = (w / scale).round().clamp(-128, 127).to(torch.int8)

    new_mod = Int8Linear(w_int8, scale, mod.bias)

    model.replace_module(module.module_name, new_mod)
    
    module.module_data = new_mod

In [6]:
class Int4Linear(nn.Module):
    def __init__(self, w_int4: torch.Tensor, scale: torch.Tensor, bias=None):
        super().__init__()
        self.register_buffer("weight", w_int4)
        self.register_buffer("scale", scale)
        self.bias = bias

    def forward(self, x):
        w = self.weight.to(x.dtype) * self.scale
        return nn.functional.linear(x, w, self.bias)


def action_quantize_int4(module: Module, model: Model) -> None:
    """
        Replace nn.Linear with Int4Linear.

        Quantizes weights to int4 range [-8, 7] stored in int8 tensors
        (PyTorch has no native int4 dtype).
    """
    mod = module.module_data
    if not isinstance(mod, nn.Linear):
        raise Exception(f"Module '{module.module_name}' is not an nn.Linear. Operation aborted.")

    w = mod.weight.data
    scale = (w.abs().max() / 7.0).to(torch.float16)  
    w_int4 = (w / scale).round().clamp(-8, 7).to(torch.int8)

    new_mod = Int4Linear(w_int4, scale, mod.bias)

    model.replace_module(module.module_name, new_mod)
    module.module_data = new_mod

In [7]:
class Int1Linear(nn.Module):
    def __init__(self, w_binary: torch.Tensor, scale: torch.Tensor, bias=None):
        super().__init__()
        self.register_buffer("weight", w_binary)
        self.register_buffer("scale", scale)
        self.bias = bias

    def forward(self, x):
        # unpack 0/1 back to -1/+1 at runtime
        w = (self.weight.to(x.dtype) * 2 - 1) * self.scale
        return nn.functional.linear(x, w, self.bias)


def action_quantize_int1(module: Module, model: Model) -> None:
    """
        Replace nn.Linear with Int1Linear.

        Binary quantization: each weight becomes -1 or +1.
        Scale is the mean absolute value of the original weights (BitNet-style).
        Bits are stored as uint8 (0 = -1, 1 = +1).
    """
    mod = module.module_data
    if not isinstance(mod, nn.Linear):
        raise Exception(f"Module '{module.module_name}' is not an nn.Linear. Operation aborted.")

    w = mod.weight.data
    scale = w.abs().mean().to(torch.float16)   # BitNet-style scale
    w_binary = (w > 0).to(torch.uint8)         # True→1 (+1), False→0 (-1)

    new_mod = Int1Linear(w_binary, scale, mod.bias)

    model.replace_module(module.module_name, new_mod)
    module.module_data = new_mod

In [8]:
def is_linear() ->  Callable[[Module], bool]:
    return lambda m: isinstance(m.module_data, nn.Linear)

def name_contains(substring: str) -> Callable[[Module], bool]:
    return lambda m: substring in m.module_name

def name_not_contains(substring: str) -> Callable[[Module], bool]:
    return lambda m: substring not in m.module_name

def is_type(t: type) -> Callable[[Module], bool]:
    return lambda m: isinstance(m.module_data, t)

def all_of(*conditions) -> Callable[[Module], bool]:
    return lambda m: all(c(m) for c in conditions)

def any_of(*conditions) -> Callable[[Module], bool]:
    return lambda m: any(c(m) for c in conditions)

In [ ]:
model_name = "Qwen/Qwen2.5-3B-Instruct"
token = os.environ.get("HF_TOKEN")
if token is None:
    raise Exception("Token for HuggingFace is not set")

model = Model("Qwen/Qwen2.5-3B-Instruct")
model.load_from_hf(token)

analyzer = Analyzer()
analyzer.collect_weights(model, reduction=0.00001, min_count=128, max_count=1024)

display = Display()
display.show_all_distribution(analyzer.get_all_weights(model), log_y=True)
display.show_layer_distributions(analyzer.get_all_module_weights(model), log_y=True)

# quantizer = Quantizer()
# quantizer.add_rule(Rule("Quantize all linear layers", condition=all_of(is_linear(), name_not_contains("lm_head")), action=action_quantize_int8))
# quantizer.apply(model)

# analyzer.collect_weights(model, reduction=0.00001)
# display.show_all_distribution(analyzer.get_all_weights(model), log_y=True)

Loading weights: 100%|██████████| 434/434 [00:00<00:00, 655.60it/s, Materializing param=model.norm.weight]                              


KeyboardInterrupt: 